In [1]:
%pip install jupysql pandas matplotlib duckdb-engine
import pandas as pd
import build_geonames_db
import geonames_db_search
from pathlib import Path
import time
from utils import format_time

if "conn" in globals():
    globals()["conn"].close() # Allow script rerunning

%load_ext sql
geonames_search = geonames_db_search.GeonamesSearch(topk=5, threshold=3)
conn = geonames_search.connection
%config SqlMagic.displaylimit = None
%sql conn --alias duckdb

Note: you may need to restart the kernel to use updated packages.


The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

displaylimit: Value None will be treated as 0 (no limit)

In [2]:

print(f"DB Size: {Path(build_geonames_db.DUCK_DB_PATH).stat().st_size / 1e9:.2f} GB")
description = conn.execute("""DESCRIBE""").fetchdf()
description = description[['database', 'name', 'column_names', 'column_types']]
description.set_index(['database', 'name'], inplace=True)
description.insert(0, 'row_count', float('nan'))


for db_name, table_name in description.index:
    if table_name not in ["fullHierarchy", "allNames"]:
        row_count = conn.execute(f'SELECT COUNT(*) FROM {db_name}.{table_name}').fetchone()[0]
        description.loc[(db_name, table_name), 'row_count'] = row_count

display(description.style.format({'row_count': '{:_.0f}'}))

feature_codes = conn.execute(
    """
        SELECT feature_class, feature_code, COUNT(*) AS count
        FROM geonames
        GROUP BY feature_class, feature_code
        ORDER BY count DESC
""").fetchdf()
feature_codes['feature_code'] = feature_codes['feature_class'] + "." + feature_codes['feature_code']
feature_codes.drop(columns=['feature_class'], inplace=True)
feature_codes["proportion"] = feature_codes['count'] / feature_codes['count'].sum()
feature_codes = feature_codes.head(20)
try:
    build_geonames_db.download_file("https://download.geonames.org/export/dump/featureCodes_en.txt", Path("dumps/geonames/featureCodes_en.txt"))
    feature_codes = feature_codes.merge(
        pd.read_csv(
            "dumps/geonames/featureCodes_en.txt", 
            sep='\t', header=None, names=["feature_code", "designation", "description"]
    ), on='feature_code', how='left')
    feature_codes = feature_codes[['feature_code', 'designation', 'count', 'proportion']]
except Exception as e:
    print(f"Could not download feature code descriptions: {e}")
display(
    feature_codes.style
    .format({'count': '{:_.0f}', 'proportion': '{:.0%}'})
    .bar(subset=['proportion'], vmin=0, vmax=1)
)

DB Size: 1.88 GB


,feature_code,designation,count,proportion
0,P.PPL,populated place,4_685_832,35%
1,H.STM,stream,990_177,7%
2,T.HLL,hill,502_096,4%
3,T.MT,mountain,422_594,3%
4,S.FRM,farm,359_812,3%
5,H.LK,lake,309_547,2%
6,S.SCH,school,298_042,2%
7,H.STMI,intermittent stream,261_178,2%
8,S.CH,church,259_635,2%
9,S.HTL,hotel,241_608,2%


In [3]:
%%sql
SELECT COUNT(*) AS number_of_cities FROM geonames WHERE feature_class = 'P'

Running query in 'duckdb'

number_of_cities
5195947


5 195 947 populated places in geonames against [2 887 930](https://query.wikidata.org/#SELECT%20%28COUNT%28DISTINCT%20%3Fid%29%20AS%20%3Fcount%29%20WHERE%20%7B%0A%20%20%3Fid%20wdt%3AP31%20%2F%20wdt%3AP279%2a%20wd%3AQ123964505.%0A%7D) on wikidata

However there are [709 681](https://query.wikidata.org/#SELECT%20%28COUNT%28DISTINCT%20%3Fid%29%20AS%20%3Fcount%29%20WHERE%20%7B%0A%20%20%3Fid%20wdt%3AP31%20%2F%20wdt%3AP279%2a%20wd%3AQ123964505.%0A%20%20%3Fid%20wdt%3AP1566%20%3FsomeGeonamesId%0A%7D) wikidata populated places that don't link to a geoname.

Among which [13 554](https://query.wikidata.org/#SELECT%20%28COUNT%28DISTINCT%20%3Fid%29%20AS%20%3Fcount%29%20WHERE%20%7B%0A%20%20%3Fid%20wdt%3AP31%20%2F%20wdt%3AP279%2a%20wd%3AQ253019.%0A%20%20%3Fid%20wdt%3AP1566%20%3FsomeGeonamesId%0A%7D) are ["Ortsteil"](https://www.wikidata.org/wiki/Q253019) including [Sudberg](https://www.wikidata.org/wiki/Q2362997) in Wuppertal, which is present in bzk open but not [geonames](https://www.geonames.org/2805753/wuppertal.html) neither gnd.

In [4]:
%%sql
SELECT levenshtein(nfc_normalize(lower(alternateName)), nfc_normalize(lower('Sudberg'))) AS score, *
FROM allNames NATURAL LEFT JOIN simplifiedGeonames
WHERE score <= 2 AND feature_class = 'P'
ORDER BY score
LIMIT 10;

Running query in 'duckdb'

score,geonameId,isolanguage,alternateName,isPreferredName,isShortName,isColloquial,isHistoric,gndUri,name,asciiname,feature_class,feature_code,country_code,admin1_code,admin2_code,admin3_code,admin4_code,admin5_code,parentCityIds,parentRegionIds
1,2679376,None,Sidberg,True,None,None,None,None,Sidberg,Sidberg,P,PPL,SE,23,2481,None,None,None,None,None
1,5214588,None,Suedberg,True,None,None,None,None,Suedberg,Suedberg,P,PPL,US,PA,107,60464,None,None,None,None
1,2824593,None,Surberg,True,None,None,None,None,Surberg,Surberg,P,PPLA4,DE,02,091,09189,09189148,None,None,None
1,2658465,None,Suberg,None,None,None,None,None,Suberg,Suberg,P,PPL,CH,BE,243,303,None,None,None,None
1,11587121,de,Surberg,None,None,None,None,None,Surberg,Surberg,P,PPLL,CH,SG,1724,3275,None,None,None,None
1,2824593,None,Surberg,True,None,None,None,https://d-nb.info/gnd/4305569-2,Surberg,Surberg,P,PPLA4,DE,02,091,09189,09189148,None,None,None
1,11962460,se,Sundberg,True,None,None,None,None,Salmenkallio,Salmenkallio,P,PPLX,FI,01,011,091,None,None,None,None
1,2658465,None,Suberg,True,None,None,None,None,Suberg,Suberg,P,PPL,CH,BE,243,303,None,None,None,None
1,11587121,None,Surberg,True,None,None,None,None,Surberg,Surberg,P,PPLL,CH,SG,1724,3275,None,None,None,None
1,2942108,None,Budberg,None,None,None,None,None,Budberg,Budberg,P,PPL,DE,07,059,05974,05974052,None,None,None


In [5]:
print("\n".join([x[1] for x in conn.execute(
    "EXPLAIN ANALYZE " +
    geonames_db_search.build_closest_matches_query(threshold=2)
        .replace("$1", "'New York'")
        .replace("$2", "['US']")
        .replace("$3", "NULL")
).fetchall()]))

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
EXPLAIN ANALYZE WITH query_prep AS (     SELECT          'New York' AS query,         nfc_normalize(query) AS nfc_query,         regexp_replace(lower(strip_accents(nfc_query)), '[^\w\s]', '', 'g') AS clean_query,         ['US'] AS country_restriction,         NULL AS abbreviation_pattern ), candidates AS(     SELECT filtered.*     FROM candidate_names AS filtered, query_prep     WHERE          query_prep.country_restriction IS NULL          OR          country_code IN query_prep.country_restriction  ), ranked_matches AS(     SELECT          query_prep.*,         nfc_alt_name,         clean_alt_name,         levenshtein(nfc_alt_name, nfc_query) AS raw_distance,         levenshtein(clean_alt_name, clean_query) AS cleaned_distance,         CASE              WHEN query_prep.abbreviation_patter

In [6]:
csv_read_args = dict(keep_default_na=False, dtype=str, na_values=[""])

bzkopen_val = pd.read_csv("open_data/bzkopen_addresses_val.csv", **csv_read_args)
ESTIMATED_TOTAL_ADDRESSES = 4_228_682 # from compare.ipynb

In [7]:
print("Benchmark complete address search, using country restrictions...")
start_time = time.monotonic()
matches = geonames_search.link_parsed_addresses(bzkopen_val)
end_time = time.monotonic()
elapsed = end_time - start_time
print(f"Search took {format_time(elapsed)}\nEstimate time for {ESTIMATED_TOTAL_ADDRESSES:_} addresses: {format_time(((elapsed)/ len(bzkopen_val)) * ESTIMATED_TOTAL_ADDRESSES)}")
display(matches[matches["query"] == "Austr."].head())
display(matches[matches["query"] == "N.Y."].head())
matches.head()

Benchmark complete address search, using country restrictions...
Country hints set for 0 / 20 addresses for entity type Country
Starting search for entity type Country
Query hits: {'1st': 0, '2nd': 0, '3rd': 20}
Search for entity type Country took 0:00:01 and returned 127 matches
Country hints set for 20 countries
Country hints set for 0 / 5 addresses for entity type State
Starting search for entity type State
Query hits: {'1st': 0, '2nd': 5, '3rd': 0}
Search for entity type State took 0:00:00 and returned 61 matches
Country hints set for 0 / 17 addresses for entity type Region
Starting search for entity type Region
Query hits: {'1st': 0, '2nd': 17, '3rd': 0}
Search for entity type Region took 0:00:02 and returned 207 matches
Country hints set for 0 / 24 addresses for entity type District
Starting search for entity type District
Query hits: {'1st': 0, '2nd': 21, '3rd': 1}
Search for entity type District took 0:00:02 and returned 257 matches
Country hints set for 27 / 111 addresses for 

query nfc_query clean_query  \
input_row entity_type entity_rank geonameId                                 
112       Country     1           2782113    Austr.    Austr.       austr   
                      2           2077456    Austr.    Austr.       austr   
                      3           453733     Austr.    Austr.       austr   
                                  8642971    Austr.    Austr.       austr   
                      5           3723988    Austr.    Austr.       austr   

                                            abbreviation_pattern nfc_alt_name  \
input_row entity_type entity_rank geonameId                                     
112       Country     1           2782113                 austr%      Austria   
                      2           2077456                 austr%    Australia   
                      3           453733                  austr%        Eesti   
                                  8642971                 austr%         Rush   
                      5           3723988                 austr%        Ayiti   

                                            clean_alt_name  raw_distance  \
input_row entity_type entity_rank geonameId                                
112       Country     1           2782113          austria             2   
                      2           2077456        australia             4   
                      3           453733             eesti             4   
                                  8642971             rush             4   
                      5           3723988            ayiti             4   

                                             cleaned_distance  \
input_row entity_type entity_rank geonameId                     
112       Country     1           2782113                   2   
                      2           2077456                   4   
                      3           453733                    3   
                                  8642971                   3   
                      5           3723988                   3   

                                             may_be_abbreviation isolanguage  \
input_row entity_type entity_rank geonameId                                    
112       Country     1           2782113                   True          en   
                      2           2077456                   True          en   
                      3           453733                   False          et   
                                  8642971                  False         NaN   
                      5           3723988                  False          ht   

                                             ... admin1_code  admin2_code  \
input_row entity_type entity_rank geonameId  ...                            
112       Country     1           2782113    ...          00         None   
                      2           2077456    ...          00         None   
                      3           453733     ...          00         None   
                                  8642971    ...           L         None   
                      5           3723988    ...          00         None   

                                             admin3_code  admin4_code  \
input_row entity_type entity_rank geonameId                             
112       Country     1           2782113           None         None   
                      2           2077456           None         None   
                      3           453733            None         None   
                                  8642971           None         None   
                      5           3723988           None         None   

                                             admin5_code parentCityIds  \
input_row entity_type entity_rank geonameId                              
112       Country     1           2782113           None          <NA>   
                      2           2077456           None          <NA>   
                      3           453733

query nfc_query clean_query  \
input_row entity_type entity_rank geonameId                               
58        State       1           5128638    N.Y.      N.Y.          ny   
                      2           293164     N.Y.      N.Y.          ny   
                      3           5073863    N.Y.      N.Y.          ny   
                      4           293197     N.Y.      N.Y.          ny   
                      5           2641276    N.Y.      N.Y.          ny   

                                            abbreviation_pattern  \
input_row entity_type entity_rank geonameId                        
58        State       1           5128638                  n% y%   
                      2           293164                   n% y%   
                      3           5073863                  n% y%   
                      4           293197                   n% y%   
                      5           2641276                  n% y%   

                                                       nfc_alt_name  \
input_row entity_type entity_rank geonameId                           
58        State       1           5128638                        NY   
                      2           293164             Nefat Yizre‘el   
                      3           5073863         New York Township   
                      4           293197         Nefat Yerushalayim   
                      5           2641276    North Riding Yorkshire   

                                                     clean_alt_name  \
input_row entity_type entity_rank geonameId                           
58        State       1           5128638                        ny   
                      2           293164              nefat yizreel   
                      3           5073863         new york township   
                      4           293197         nefat yerushalayim   
                      5           2641276    north riding yorkshire   

                                             raw_distance  cleaned_distance  \
input_row entity_type entity_rank geonameId                                   
58        State       1           5128638               2                 0   
                      2           293164               14                11   
                      3           5073863              15                15   
                      4           293197               16                16   
                      5           2641276              20                20   

                                             may_be_abbreviation isolanguage  \
input_row entity_type entity_rank geonameId                                    
58        State       1           5128638                  False        abbr   
                      2           293164                    True         NaN   
                      3           5073863                   True          en   
                      4           293197                    True         NaN   
                      5           2641276                   True         NaN   

                                             ... admin1_code  admin2_code  \
input_row entity_type entity_rank geonameId  ...                            
58        State       1           5128638    ...          NY          NaN   
                      2           293164     ...          03          NaN   
                      3           5073863    ...          NE          185   
                      4           293197     ...          06          NaN   
                      5           2641276    ...         ENG          NaN   

                                             admin3_code  admin4_code  \
input_row entity_type entity_rank geonameId                             
58        State       1           5128638           None         None   
                      2           293164            None         None   
                      3           5073863           None         None   
                      4  

query   nfc_query  \
input_row entity_type entity_rank geonameId                           
0         City        1           2849483    Regensburg  Regensburg   
                      2           2659084    Regensburg  Regensburg   
                                  2849484    Regensburg  Regensburg   
                      4           2767442    Regensburg  Regensburg   
                                  2767443    Regensburg  Regensburg   

                                            clean_query abbreviation_pattern  \
input_row entity_type entity_rank geonameId                                    
0         City        1           2849483    regensburg                 <NA>   
                      2           2659084    regensburg                 <NA>   
                                  2849484    regensburg                 <NA>   
                      4           2767442    regensburg                 <NA>   
                                  2767443    regensburg                 <NA>   

                                            nfc_alt_name clean_alt_name  \
input_row entity_type entity_rank geonameId                               
0         City        1           2849483     Regensburg     regensburg   
                      2           2659084     Regensberg     regensberg   
                                  2849484     Regensberg     regensberg   
                      4           2767442    Riegersburg    riegersburg   
                                  2767443    Riegersburg    riegersburg   

                                             raw_distance  cleaned_distance  \
input_row entity_type entity_rank geonameId                                   
0         City        1           2849483               0                 0   
                      2           2659084               1                 1   
                                  2849484               1                 1   
                      4           2767442               2                 2   
                                  2767443               2                 2   

                                             may_be_abbreviation isolanguage  \
input_row entity_type entity_rank geonameId                                    
0         City        1           2849483                  False         NaN   
                      2           2659084                  False         NaN   
                                  2849484                  False         NaN   
                      4           2767442                  False         NaN   
                                  2767443                  False         NaN   

                                             ... admin1_code  admin2_code  \
input_row entity_type entity_rank geonameId  ...                            
0         City        1           2849483    ...          02          093   
                      2           2659084    ...          ZH          104   
                                  2849484    ...          02          094   
                      4           2767442    ...          03          310   
                                  2767443    ...          06          623   

                                             admin3_code  admin4_code  \
input_row entity_type entity_rank geonameId                             
0         City        1           2849483          09362     09362000   
                      2           2659084             95          NaN   
                                  2849484          09474     09474145   
                      4           2767442          31016          NaN   
                                  2767443          62386          NaN   

                                             admin5_code parentCityIds  \
input_row entity_type entity_rank geonameId                              
0         City        1           2849483           None          <NA>   
                      2           2659084           None          <NA>   
  

In [8]:
abbrevs_mask = matches["abbreviation_pattern"].isna()
abbrev_time = matches[abbrevs_mask]["search_time"].mean()
non_abbrev_time = matches[~abbrevs_mask]["search_time"].mean()
print(f"Average search time for abbreviations: {format_time(abbrev_time)}")
print(f"Average search time for non-abbreviations: {format_time(non_abbrev_time)}")

Average search time for abbreviations: 0:00:00
Average search time for non-abbreviations: 0:00:00


In [9]:
start_time = time.monotonic()
all_matches = []
for entity_type in geonames_db_search.EntityType:
    print(f"Benchmarking {entity_type.name} matches...")
    names = bzkopen_val[entity_type.name].dropna().unique()
    sub_start_time = time.monotonic()
    matches_ = geonames_search.link_entities(names, entity_type=entity_type)
    sub_end_time = time.monotonic()
    all_matches.extend(matches_)
    print(f"  {entity_type.name}: {len(matches_)} matches (took {format_time(sub_end_time - sub_start_time)})")
end_time = time.monotonic()
elapsed = end_time - start_time
print(f"Search took {format_time(elapsed)}\nEstimate time for {ESTIMATED_TOTAL_ADDRESSES:_} addresses: {format_time(((elapsed)/ len(bzkopen_val)) * ESTIMATED_TOTAL_ADDRESSES)}")
all_matches = pd.concat(all_matches, ignore_index=True)
display(all_matches[all_matches["query"] == "Austr."])
display(all_matches[all_matches["query"] == "N.Y."])
all_matches

Benchmarking Country matches...
Query hits: {'1st': 0, '2nd': 0, '3rd': 20}
  Country: 20 matches (took 0:00:02)
Benchmarking State matches...
Query hits: {'1st': 0, '2nd': 4, '3rd': 0}
  State: 4 matches (took 0:00:01)
Benchmarking Region matches...
Query hits: {'1st': 0, '2nd': 17, '3rd': 0}
  Region: 17 matches (took 0:00:02)
Benchmarking District matches...
Query hits: {'1st': 0, '2nd': 21, '3rd': 1}
  District: 24 matches (took 0:00:02)
Benchmarking City matches...
Query hits: {'1st': 0, '2nd': 103, '3rd': 1}
  City: 108 matches (took 0:00:22)
Benchmarking Neighborhood matches...
Query hits: {'1st': 0, '2nd': 14, '3rd': 0}
  Neighborhood: 16 matches (took 0:00:04)
Search took 0:00:33
Estimate time for 4_228_682 addresses: 10 days, 18:33:28


,query,nfc_query,clean_query,country_restriction,abbreviation_pattern,nfc_alt_name,clean_alt_name,raw_distance,cleaned_distance,may_be_abbreviation,...,admin2_code,admin3_code,admin4_code,admin5_code,parentCityIds,parentRegionIds,Country,entity_type_map,entity_rank,search_time
81,Austr.,Austr.,austr,<NA>,Austr%,Austria,austria,2,2,False,...,None,None,None,None,<NA>,"[12718413, 12503661, 9408659, 12217848]",Austria,"{'Country': True, 'State': False, 'Region': Fa...",1,0.037101
82,Austr.,Austr.,austr,<NA>,Austr%,Eesti,eesti,4,3,False,...,None,None,None,None,<NA>,"[12718413, 12503661, 7729883, 12217933, 12217934]",Estonia,"{'Country': True, 'State': False, 'Region': Fa...",2,0.037101
83,Austr.,Austr.,austr,<NA>,Austr%,Rush,rush,4,3,False,...,None,None,None,None,<NA>,<NA>,Ireland,"{'Country': True, 'State': False, 'Region': Fa...",2,0.037101
84,Austr.,Austr.,austr,<NA>,Austr%,Ayiti,ayiti,4,3,False,...,None,None,None,None,<NA>,[7729891],Haiti,"{'Country': True, 'State': False, 'Region': Fa...",4,0.037101
85,Austr.,Austr.,austr,<NA>,Austr%,Cssr,cssr,4,3,False,...,None,None,None,None,<NA>,<NA>,Serbia and Montenegro,"{'Country': True, 'State': False, 'Region': Fa...",4,0.037101


,query,nfc_query,clean_query,country_restriction,abbreviation_pattern,nfc_alt_name,clean_alt_name,raw_distance,cleaned_distance,may_be_abbreviation,...,admin2_code,admin3_code,admin4_code,admin5_code,parentCityIds,parentRegionIds,Country,entity_type_map,entity_rank,search_time
124,N.Y.,N.Y.,ny,<NA>,n% y%,NY,ny,2,0,False,...,NaN,None,None,None,<NA>,"[12902582, 12746282, 11887749, 12212303, 12212...",United States,"{'Country': False, 'State': True, 'Region': Tr...",1,0.067319
125,N.Y.,N.Y.,ny,<NA>,n% y%,Nefat Yizre‘el,nefat yizreel,14,11,True,...,NaN,None,None,None,<NA>,<NA>,Israel,"{'Country': False, 'State': True, 'Region': Tr...",2,0.067319
126,N.Y.,N.Y.,ny,<NA>,n% y%,New York Township,new york township,15,15,True,...,185,None,None,None,<NA>,<NA>,United States,"{'Country': False, 'State': True, 'Region': Tr...",3,0.067319
127,N.Y.,N.Y.,ny,<NA>,n% y%,Nefat Yerushalayim,nefat yerushalayim,16,16,True,...,NaN,None,None,None,<NA>,<NA>,Israel,"{'Country': False, 'State': True, 'Region': Tr...",4,0.067319
128,N.Y.,N.Y.,ny,<NA>,n% y%,North Riding Yorkshire,north riding yorkshire,20,20,True,...,NaN,None,None,None,<NA>,<NA>,United Kingdom,"{'Country': False, 'State': True, 'Region': Tr...",5,0.067319
1780,N.Y.,N.Y.,ny,<NA>,n% y%,NY,ny,2,0,False,...,NaN,NaN,NaN,None,<NA>,[12213808],United States,"{'Country': False, 'State': False, 'Region': F...",1,1.130774
1781,N.Y.,N.Y.,ny,<NA>,n% y%,Ny,ny,3,0,False,...,WLX,83,83028,None,<NA>,<NA>,Belgium,"{'Country': False, 'State': False, 'Region': F...",2,1.130774
1782,N.Y.,N.Y.,ny,<NA>,n% y%,New Yatt,new yatt,6,6,True,...,K2,38UF,38UF039,None,<NA>,<NA>,United Kingdom,"{'Country': False, 'State': False, 'Region': F...",3,1.130774
1783,N.Y.,N.Y.,ny,<NA>,n% y%,New York,new york,6,6,True,...,006,NaN,NaN,None,<NA>,<NA>,United States,"{'Country': False, 'State': False, 'Region': F...",3,1.130774
1784,N.Y.,N.Y.,ny,<NA>,n% y%,New York,new york,6,6,True,...,185,94281,NaN,None,<NA>,<NA>,United States,"{'Country': False, 'State': False, 'Region': F...",3,1.130774


,query,nfc_query,clean_query,country_restriction,abbreviation_pattern,nfc_alt_name,clean_alt_name,raw_distance,cleaned_distance,may_be_abbreviation,...,admin2_code,admin3_code,admin4_code,admin5_code,parentCityIds,parentRegionIds,Country,entity_type_map,entity_rank,search_time
0,England,England,england,<NA>,<NA>,England,england,0,0,False,...,None,None,None,None,<NA>,<NA>,United Kingdom,"{'Country': True, 'State': True, 'Region': Tru...",1,0.115363
1,England,England,england,<NA>,<NA>,Estland,estland,2,2,False,...,None,None,None,None,<NA>,"[12718413, 12503661, 7729883, 12217933, 12217934]",Estonia,"{'Country': True, 'State': False, 'Region': Fa...",2,0.115363
2,England,England,england,<NA>,<NA>,Eisland,eisland,2,2,False,...,None,None,None,None,<NA>,"[12503661, 2616167, 9961132, 7729883]",Iceland,"{'Country': True, 'State': False, 'Region': Fa...",3,0.115363
3,England,England,england,<NA>,<NA>,Indland,indland,2,2,False,...,None,None,None,None,<NA>,[7729895],India,"{'Country': True, 'State': False, 'Region': Fa...",3,0.115363
4,England,England,england,<NA>,<NA>,Poland,poland,3,3,False,...,None,None,None,None,<NA>,"[12503661, 7729884, 12217933, 12217848]",Poland,"{'Country': True, 'State': False, 'Region': Fa...",5,0.115363
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2742,Jackson Heights,Jackson Heights,jackson heights,<NA>,<NA>,Jackson Heights,jackson heights,0,0,False,...,119,NaN,None,None,<NA>,<NA>,United States,"{'Country': False, 'State': False, 'Region': F...",1,0.237376
2743,Jackson Heights,Jackson Heights,jackson heights,<NA>,<NA>,Jackson Heights,jackson heights,0,0,False,...,081,NaN,None,None,<NA>,<NA>,United States,"{'Country': False, 'State': False, 'Region': F...",1,0.237376
2744,Jackson Heights,Jackson Heights,jackson heights,<NA>,<NA>,Jackson Heights,jackson heights,0,0,False,...,063,NaN,None,None,<NA>,<NA>,United States,"{'Country': False, 'State': False, 'Region': F...",1,0.237376
2745,Jackson Heights,Jackson Heights,jackson heights,<NA>,<NA>,Jackson Heights,jackson heights,0,0,False,...,081,80878,None,None,<NA>,<NA>,United States,"{'Country': False, 'State': False, 'Region': F...",1,0.237376


In [10]:
#query = geonames_db_search.build_closest_matches_query(entity_type=geonames_db_search.EntityType["City"], topk=10, threshold=3)
#query = query.replace("$1", "'Karlsruhe'")
#query = query.replace("$2", "NULL")
#result = conn.execute("EXPLAIN ANALYZE " + query).fetchall()
#for row in result:
#    print(row[1])

In [11]:
%%sql
SELECT * FROM allNames WHERE geonameId = 2267057;

Running query in 'duckdb'

geonameId,isolanguage,alternateName,isPreferredName,isShortName,isColloquial,isHistoric,gndUri
2267057,hu,Lisszabon,None,None,None,None,None
2267057,la,Lisbonum,None,None,None,None,None
2267057,af,Lissabon,None,None,None,None,None
2267057,fi,Lissabon,None,None,None,None,None
2267057,unlc,PTLIS,None,None,None,None,None
2267057,it,Lisbona,None,None,None,None,None
2267057,gsw,Lissabon,None,None,None,None,None
2267057,br,Lisbon,None,None,None,None,None
2267057,ro,Lisabona,None,None,None,None,None
2267057,sc,Lisbona,None,None,None,None,None


In [12]:
%%sql
SELECT levenshtein(nfc_normalize(lower(alternateName)), nfc_normalize(lower('N.Y.'))) AS distance, (CASE WHEN alternateName SIMILAR TO 'Austr\w+' THEN 0 ELSE distance END) AS score, *
FROM allNames NATURAL LEFT JOIN simplifiedGeonames
WHERE (score <= 2) AND feature_class = 'A'
ORDER BY score
LIMIT 10;

Running query in 'duckdb'

distance,score,geonameId,isolanguage,alternateName,isPreferredName,isShortName,isColloquial,isHistoric,gndUri,name,asciiname,feature_class,feature_code,country_code,admin1_code,admin2_code,admin3_code,admin4_code,admin5_code,parentCityIds,parentRegionIds
7,0,7296061,None,Austrey,True,None,None,None,None,Austrey,Austrey,A,ADM4,GB,ENG,P3,44UB,44UB005,None,None,[7290646]
7,0,2782113,qu,Austria,True,None,None,None,None,Republic of Austria,Republic of Austria,A,PCLI,AT,00,None,None,None,None,None,"[12718413, 12503661, 9408659, 12217848]"
13,0,1966436,lv,Austrumtimora,True,None,None,None,None,Democratic Republic of Timor-Leste,Democratic Republic of Timor-Leste,A,PCLI,TL,00,None,None,None,None,None,"[7729896, 12217089]"
7,0,2782113,ms,Austria,True,None,None,None,None,Republic of Austria,Republic of Austria,A,PCLI,AT,00,None,None,None,None,None,"[12718413, 12503661, 9408659, 12217848]"
7,0,2782113,rm,Austria,True,None,None,None,None,Republic of Austria,Republic of Austria,A,PCLI,AT,00,None,None,None,None,None,"[12718413, 12503661, 9408659, 12217848]"
15,0,2789733,lv,Austrumflandrija,None,None,None,None,None,Provincie Oost-Vlaanderen,Provincie Oost-Vlaanderen,A,ADM2,BE,VLG,VOV,None,None,None,None,[3337388]
8,0,2782113,hr,Austrija,True,None,None,None,None,Republic of Austria,Republic of Austria,A,PCLI,AT,00,None,None,None,None,None,"[12718413, 12503661, 9408659, 12217848]"
7,0,2782113,ro,Austria,True,None,None,None,None,Republic of Austria,Republic of Austria,A,PCLI,AT,00,None,None,None,None,None,"[12718413, 12503661, 9408659, 12217848]"
7,0,2782113,sn,Austria,True,None,None,None,None,Republic of Austria,Republic of Austria,A,PCLI,AT,00,None,None,None,None,None,"[12718413, 12503661, 9408659, 12217848]"
8,0,2782113,lv,Austrija,True,None,None,None,None,Republic of Austria,Republic of Austria,A,PCLI,AT,00,None,None,None,None,None,"[12718413, 12503661, 9408659, 12217848]"


In [13]:
%%sql
SELECT parent.name AS parentName, parentId, child.name as childName, childId 
FROM hierarchy 
    JOIN geonames AS parent ON (hierarchy.parentId = parent.geonameId)
    JOIN geonames AS child ON (hierarchy.childId = child.geonameId)
WHERE 
    parent.country_code = child.country_code IS NOT TRUE AND
    parent.admin1_code = child.admin1_code IS NOT TRUE AND
    parent.admin2_code = child.admin2_code IS NOT TRUE AND
    parent.admin3_code = child.admin3_code IS NOT TRUE AND
    parent.admin4_code = child.admin4_code IS NOT TRUE AND
    parent.admin5_code = child.admin5_code IS NOT TRUE AND
    hierarchy.type = 'ADM'
LIMIT 10;

Running query in 'duckdb'

parentName,parentId,childName,childId
Earth,6295630,Africa,6255146
Earth,6295630,Asia,6255147
Earth,6295630,Europe,6255148
Earth,6295630,North America,6255149
Earth,6295630,South America,6255150
Earth,6295630,Oceania,6255151
Earth,6295630,Antarctica,6255152
Africa,6255146,Republic of Rwanda,49518
Asia,6255147,Macau Special Administrative Region,1821275
Oceania,6255151,Commonwealth of the Northern Mariana Islands,4041468


In [14]:
%%sql
WITH 
geonames1 AS (
    SELECT * 
    FROM geonames
    WHERE geonames.geonameId IN (7117596, 5844411)
),
adm2 AS (
    SELECT geonames.*
    FROM geonames 
        RIGHT JOIN geonames1 USING (country_code, admin1_code, admin2_code) 
    WHERE 
        geonames.feature_class = 'A' AND 
        geonames.feature_code IN ('ADM2', 'ADM2H')
),
adm1 AS (
    SELECT geonames.*
    FROM geonames 
        RIGHT JOIN geonames1 USING (country_code, admin1_code) 
    WHERE 
        geonames.feature_class = 'A' AND 
        geonames.feature_code IN ('ADM1', 'ADM1H')
),
country AS (
    SELECT geonames.* 
    FROM geonames 
        JOIN countryInfo ON (geonames.geonameId = countryInfo.geonameid) 
        JOIN geonames1 USING (country_code)
)
SELECT * FROM geonames1 
UNION 
SELECT * FROM adm2
UNION
SELECT * FROM adm1
UNION
SELECT * FROM country

Running query in 'duckdb'

geonameId,name,asciiname,alternatenames,latitude,longitude,feature_class,feature_code,country_code,cc2,admin1_code,admin2_code,admin3_code,admin4_code,admin5_code,population,elevation,dem,timezone,modification_date
5844411,Annette Island Reserve,Annette Island Reserve,"Annette Island Indian Reservation,Annette Island Reserve",55.12839889526367,-131.4965362548828,A,ADMD,US,None,AK,198,None,None,None,5559,63,97,America/Metlakatla,2017-05-25
5879092,Alaska,Alaska,"'Alaka,AK,Alaeksa,Alakskak,Alaksu,Alasca,Alaschka,Alaska,Alaska suyu,Alaskaa,Alasko,Alaszka,Aleska,Aliaksa,Aliaska,Aljask,Aljaska,Aljaska shtaty,Aljaskae,Aljaske,Aljaška,Alyaska,Alyaska Shitati,Alâska,Alýaska,Anian,Aļaska,Ałasca,Estado de Alaska,Gamaland,Hakʼaz Dineʼé Bikéyah,I-Alaskha,Land of the Midnight Sun,New Albion,Sewards Folly,State of Alaska,Territory of Alaska,a la si jia,a la si jia zhou,alasaka,alaska,allaeseuka ju,allaeseukaju,arasuka zhou,yalaska,Àláskà,Αλάσκα,Аласка,Аляск,Аляскæ,Аляска,Аляска штаты,Аляске,Алјаска,Аљаска,Ալասքա,Ալյասկա,אלאסקע,אלסקה,آلاسکا,ألاسكا,ئالاسکا,الاسكا,الاسکا,الاسڪا,ܐܠܐܣܟܐ,अलास्का,अलास्‍का,আলাস্কা,ਅਲਾਸਕਾ,અલાસ્કા,ଆଲାସ୍କା,அலாஸ்கா,అలాస్కా,ಅಲಾಸ್ಕ,അലാസ്ക,ඇලස්කාව,รัฐอะแลสกา,အလက်စကာပြည်နယ်,ალასკა,አላስካ,ᎠᎳᏍᎦ,ᐊᓛᓯᑲ,ᱟᱞᱟᱥᱠᱟ,‘Ālaka,ⴰⵍⴰⵙⴽⴰ,アラスカ州,阿拉斯加,阿拉斯加州,ꯑꯂꯥꯁꯀꯥ,알래스카 주,알래스카주,𐌰𐌻𐌰𐍃𐌺𐌰",64.00028228759766,-150.00027465820312,A,ADM1,US,None,AK,None,None,None,None,740133,421,425,America/Anchorage,2025-04-12
6252001,United States,United States,"'Amelika,A.S,ABD,ABS,ABŞ,ACSH,AISH,AKSH,ALeaa-Oko ti Amerika,AMN,ANU,AQSH,AS,Amelika,America,Amerika,Amerika Birlesik Devletleri,Amerika Birleşik Devletleri,Amerika Sarikat,Amerikan Yhdysvallat,Amerikas Foerenta Stater,Amerikas Forenede Stater,Amerikas Förenta Stater,Amerikas Savienotas Valstis,Amerikas Savienotās Valstis,Amerikas forente stater,Amerikayi Miacʻeal Nahangner,Amerikayi Miacʻyal Nahangner,Ameriketako Estatu Batuak,Ameriki,Amurka,Amérika Sarikat,Amɛrika,Amẹrikà,Bandariki Nordur-Ameriku,Bandarikin,Bandaríki Norður-Ameríku,Bandaríkin,DYA,Dei amerikanske sambandsstatane,Dewleten Yekbuyi yen Amerikaye,Dewletên Yekbûyî yên Amerîkayê,Dowlaaji Dentuɗi Amerik,EUA,Estados Unidos,Estats Units,Estatu Batuak,Etaa Sini,Etats-Unis,Etazonia,Feriene Steaten,Forente stater,Hoa Ky,Hoa Kỳ,Hononga o Amerika,IM,IPA,JAV,Jungtines Amerikos Valstijos,Jungtines Valstijos,Jungtinės Amerikos Valstijos,Jungtinės Valstijos,Leta Zunze Ubumwe za Amerika,Maraykanka,Marekani,Miacʻeal Nahangner,Qo'shma Shtatlar,S.U.A.,SA,SAD,SAM,SASHH,SHBA,SSHA,SU,SUA,Sambandsstatane,Sjedinene Americke Drzave,Sjedinene Drzave,Sjedinjene Americke Drzave,Sjedinjene Američke Države,Soedinennye Shtaty,Spojene staty,Spojene staty americke,Spojené státy,Spojené státy americké,Spojené štáty,Spojené štáty americké,Spoluceni Stati Ameriki,Stadis Unids da l'America,Stadis Unids da l’America,Stany Zjednoczone,Statele Unite ale Americii,Stati Uniti,Statys Unys,Suedineni shhati,U.S.,U.S.A.,UDA,US,USA,USA nutome,United States,United States of America,Usono,VS,VSA,Vereenegt Staaten,Vereinigte Staaten,Vereinigten Staaten von Amerika,Verenigde Staten,Yhdysvallat,ZDA,Zdruzene drzave Amerike,Združene države Amerike,Zlucanya Staty Ameryki,Zluchanyja Shtaty,alawlayat almthdt alamrykyt,alwlayat almthdt alamrykyt,amerika,amryka,ashsh,ayalat mthdh,ayalat mthdh amryka,i'u esa,i-U.S,l-Istati Uniti,ma. yu.,mei guo,migug,mthdh ayalat,ps,sanyukta rajya amarika,sanyukta rajya amerika,sanyukta rajya:,shrath‡,wڵatە yەkgrtwwەkan,ya q sh,yu ʿesi,yu'es,yu.aisa.,yu.es,yu.es.,yu.esa.,yuktarastra,yuuaaatit,yuʿesi,yways,ywnayٹiڑ siٹyٹis,ÂLeaa-Ôko tî Amerika,États-Unis,ʻAmelika,ʼrh״b,ΗΠΑ,АИШ,АКШ,АНУ,АЦШ,АҚШ,Злучаныя Штаты,Злучаныя Штаты Амерыкі,ИМ,САД,САЩ,США,Соединенные Штаты,Сполучені Штати Америки,Съединени щати,Сједињене Америчке Државе,Сједињене Државе,ԱՄՆ,Ամերիկայի Միացեալ Նահանգներ,Ամերիկայի Միացյալ Նահանգներ,Միացեալ Նահանգներ,Միացյալ Նահանգներ,ארה״ב,פֿש,ئا ق ش,الاولايات المتحدة الامريكية,الولايات المتحدة الأمريكية,امریکا,ایالات متحده,ایالات متحدهٔ امریکا,متحده آيالات,وڵاتە یەکگرتوو

In [15]:
%%sql
SELECT COUNT(DISTINCT gnd.geonameId) FROM gndNames NATURAL JOIN gnd
WHERE NOT EXISTS (
    SELECT 1 FROM alternateNames
    WHERE alternateNames.alternateName = gndNames.name AND alternateNames.geonameId = gnd.geonameId
)
LIMIT 10;

Running query in 'duckdb'

count(DISTINCT gnd.geonameId)
45838


There are ADMD entities that are the only reference of admin1_code. For now, ignore.

In [16]:
%%sql
SELECT country.Country, geonames.* FROM geonames 
    JOIN (
        SELECT child.country_code as country_code, ANY_VALUE(child.geonameId) AS chosen_one FROM geonames AS child
        WHERE 
            starts_with(child.feature_code, 'ADMD') AND 
            child.admin1_code IS NOT NULL AND NOT EXISTS(
                SELECT * FROM geonames AS parent 
                WHERE 
                    parent.country_code = child.country_code AND
                    parent.admin1_code = child.admin1_code AND 
                    starts_with(parent.feature_code, 'ADM1')
            )
        GROUP BY child.country_code
    ) ON chosen_one = geonames.geonameId
    JOIN countryInfo AS country ON geonames.country_code = country.ISO


Running query in 'duckdb'

Country,geonameId,name,asciiname,alternatenames,latitude,longitude,feature_class,feature_code,country_code,cc2,admin1_code,admin2_code,admin3_code,admin4_code,admin5_code,population,elevation,dem,timezone,modification_date
Iraq,6687112,Ostān-e Kūt,Ostan-e Kut,"astan kwt,استان کوت",32.899200439453125,46.32870101928711,A,ADMD,IQ,None,00,None,None,None,None,0,None,62,Asia/Baghdad,2008-02-09
Iran,66873,Sanandaj,Sanandaj,"Sanandaj,Shahrestan-e Sanandaj,Shahrestān-e Sanandaj,sanandaj,shahristani sanandaj,سَنَندَج,شَهرِستانِ سَنَندَج",32.41667175292969,47.0,A,ADMD,IR,None,00,None,None,None,None,0,None,32,Asia/Tehran,2024-07-23
Germany,2856312,Ostfalen,Ostfalen,"Eastfalen,Eastphalia,Estfalia,Oostfalen,Ostfaal,Ostfaalen,Ostfalai [a. 775],Ostfalen,Ostfalia,Ostfalija,Ostfalya,Ostfaolen,Oſtfalai [a. 775],Остфалия",52.5,10.833330154418945,A,ADMDH,DE,None,00,None,None,None,None,0,None,76,Europe/Berlin,2016-03-15
Laos,1653832,Muang Phrabatphônsan,Muang Phrabatphonsan,"Muang Phrabatphonsan,Muang Phrabatphônsan,Phone Sanh",18.278850555419922,103.17266845703125,A,ADMD,LA,None,00,None,None,None,None,0,None,184,Asia/Vientiane,2020-06-10
South Korea,1832450,Yŏnghŭng-myŏn,Yonghung-myon,"Reiko-men,Reikō-men,Reko,Rēkō,Yonghung-myon,Yonghyeon-myeon,Yŏnghŭng-myŏn",37.25,126.41667175292969,A,ADMD,KR,None,00,None,None,None,None,0,None,-9999,Asia/Seoul,2013-09-21
Philippines,8367019,Bangsamoro Autonomous Region of Muslim Mindanao,Bangsamoro Autonomous Region of Muslim Mindanao,"ARMM,Autonomous Region of Muslim Mindanao,BARMM,Bangsamoro Autonomous Region of Muslim Mindanao",5.976180076599121,121.3868408203125,A,ADMD,PH,None,00,None,None,None,None,0,None,129,Asia/Manila,2020-03-10
Poland,753867,Zamojskie Województwo,Zamojskie Wojewodztwo,None,50.75,23.25,A,ADMD,PL,None,00,None,None,None,None,0,None,205,Europe/Warsaw,2007-09-10
Indonesia,1213517,Kabupaten Tapanuli Utara,Kabupaten Tapanuli Utara,"Daerah Tingkat II Tapanuli Utara,Kabupaten Tapanuli Utara",2.25,98.75,A,ADMD,ID,None,00,None,None,None,None,0,None,1405,Asia/Jakarta,2012-01-17
Myanmar,8238781,Wa Self-Administered Division,Wa Self-Administered Division,None,22.655860900878906,98.90727233886719,A,ADMD,MM,None,00,None,None,None,None,0,None,1412,Asia/Yangon,2012-04-10
Malta,9632628,Gozo and Comino District,Gozo and Comino District,None,36.04718017578125,14.249730110168457,A,ADMD,MT,None,00,None,None,None,None,0,None,48,Europe/Malta,2014-10-04


There are places under multiple cities in the informal hierachy. For now, include all.

In [17]:
%%sql
SELECT 
    childId AS childId,
    child.name AS childName,
    unnest(list(parentId)) AS parentId,
    unnest(list(parent.name)) AS parentName
FROM hierarchy
JOIN geonames AS parent ON hierarchy.parentId = parent.geonameId
JOIN geonames AS child ON hierarchy.childId = child.geonameId
WHERE 
    parent.feature_class = 'P' AND parent.feature_code != 'PPLX' AND
    hierarchy.type IS DISTINCT FROM 'ADM'
GROUP BY childId, child.name
HAVING COUNT(*) > 1;

Running query in 'duckdb'

childId,childName,parentId,parentName
634928,Tapanila,632453,Vantaa
634928,Tapanila,12750226,North Helsinki
643032,Pakila,632453,Vantaa
643032,Pakila,12750226,North Helsinki
641374,Pitäjänmäki,658225,Helsinki
641374,Pitäjänmäki,12750226,North Helsinki
865291,Triandría,734643,Pylaía
865291,Triandría,734077,Thessaloníki
2149799,Seville,2158177,Melbourne
2149799,Seville,2144794,Wandin Yallock


In [18]:
%%sql
SELECT childId, child.name AS childName, child.feature_code, parentId, parent.name AS parentName, parent.admin1_code, parent.admin2_code, parent.feature_code as parentFeatureCode, level FROM fullHierarchy 
    JOIN geonames AS parent ON fullHierarchy.parentId = parent.geonameId 
    JOIN geonames AS child ON fullHierarchy.childId = child.geonameId
WHERE fullHierarchy.childId = 9408119

Running query in 'duckdb'

childId,childName,feature_code,parentId,parentName,admin1_code,admin2_code,parentFeatureCode,level
9408119,Main Ridge,PPLX,2077456,Commonwealth of Australia,00,None,PCLI,Country
9408119,Main Ridge,PPLX,2145234,State of Victoria,07,None,ADM1,ADM1
9408119,Main Ridge,PPLX,2158177,Melbourne,07,24600,PPLA,City
9408119,Main Ridge,PPLX,2166453,Flinders,07,25340,PPL,City
9408119,Main Ridge,PPLX,7839813,Mornington Peninsula,07,25340,ADM2,ADM2


In [19]:
%%sql
SELECT * FROM fullHierarchy JOIN allNames ON fullHierarchy.parentId = allNames.geonameId 
WHERE fullHierarchy.childId = 9408119 
LIMIT 10

Running query in 'duckdb'

childId,parentId,level,geonameId,isolanguage,alternateName,isPreferredName,isShortName,isColloquial,isHistoric,gndUri
9408119,2077456,Country,2077456,sco,Australie,None,None,None,None,None
9408119,2158177,City,2158177,gre,Μελβούρνη,None,None,None,None,None
9408119,2077456,Country,2077456,ru,Австралия,True,None,None,None,None
9408119,2077456,Country,2077456,lt,Australija,True,None,None,None,None
9408119,2145234,ADM1,2145234,fy,Victoria,None,None,None,None,None
9408119,2145234,ADM1,2145234,ms,Victoria,None,None,None,None,None
9408119,2145234,ADM1,2145234,hbs,Victoria,None,None,None,None,None
9408119,2158177,City,2158177,hbs,Melbourne,None,None,None,None,None
9408119,2158177,City,2158177,ko,멜버른,True,None,None,None,None
9408119,2077456,Country,2077456,hsb,Awstralska,None,None,None,None,None
